In [1]:
import torch
from torch.utils.data import Dataset

PAD_ID = 2

def flatten_state(cur_seq):
    flat = []
    for seq in cur_seq:
        for b in seq:
            flat.append(1 if b > 0 else 0)
    return torch.tensor(flat, dtype=torch.long)

class DPODataset(Dataset):
    def __init__(self, samples):
        self.data = []
        for seq, chosen, rejected in samples:
            x = flatten_state(seq)
            self.data.append((x, chosen, rejected))

    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i]

In [2]:
import torch.nn as nn
import torch.nn.functional as F

class PolicyOnlyTransformer(nn.Module):
    def __init__(
        self,
        vocab_size=3,         # tokens {0,1,PAD_ID}
        pad_id=PAD_ID,
        max_len=4096,
        d_model=256,
        nhead=8,
        nlayers=4,
        dim_ff=512,
        dropout=0.1,
    ):
        super().__init__()
        self.pad_id = pad_id
        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos_emb   = nn.Embedding(max_len, d_model)  # learned positional embeddings

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, activation="gelu", batch_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=nlayers)
        self.norm    = nn.LayerNorm(d_model)
        self.head    = nn.Linear(d_model, 2)  # logits for {0,1}

    def forward(self, x, pad_mask):
        """
        x:        [B, T] int64 in {0,1,pad_id}
        pad_mask: [B, T] bool, True where PAD (ignored by attention)
        """
        B, T = x.size()
        pos = torch.arange(T, device=x.device).unsqueeze(0).expand(B, T)  # [B, T]
        h = self.token_emb(x) + self.pos_emb(pos)                         # [B, T, D]

        # Mask pads inside the encoder
        h = self.encoder(h, src_key_padding_mask=pad_mask)                # [B, T, D]

        # Pool from the last non-pad token per sequence
        valid = ~pad_mask                                # True where real token
        lengths = valid.long().sum(dim=1) - 1            # index of last real token
        lengths = lengths.clamp(min=0)                   # guard against all-pad (shouldn't happen)
        idx = lengths.view(B, 1, 1).expand(B, 1, h.size(-1))
        last_h = h.gather(1, idx).squeeze(1)             # [B, D]

        logits = self.head(self.norm(last_h))            # [B, 2]
        return logits

In [3]:
import torch
import torch.nn.functional as F

@torch.no_grad()
def predict_next_bit_from_prefix(model, cur_seq, device=None, temperature=1.0, strategy="argmax"):
    """
    cur_seq: list of sequences, e.g. [X, Y, Z, W], each a list/1D tensor of ±1
    returns: dict(pred in {0,1}, conf, probs[2], logits[2])
    """
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model.eval().to(device)

    # flatten to 0/1 token stream
    bits = flatten_state(cur_seq)                     # 1D LongTensor of 0/1
    T = bits.numel()
    max_len = model.pos_emb.num_embeddings

    if T > max_len:
        raise ValueError(f"prefix length {T} exceeds model max_len {max_len}")

    # build one-row padded input and mask
    x = torch.full((1, T if T > 0 else 1), PAD_ID, dtype=torch.long)  # allow empty prefix
    if T > 0:
        x[0, :T] = bits
    pad_mask = (x == PAD_ID)

    x, pad_mask = x.to(device), pad_mask.to(device)
    logits = model(x, pad_mask=pad_mask).squeeze(0)   # [2]

    if temperature != 1.0:
        logits = logits / temperature
    probs = torch.softmax(logits, dim=-1)

    if strategy == "sample":
        pred = int(torch.multinomial(probs, 1).item())
    else:
        pred = int(probs.argmax().item())

    return {
        "pred":   pred,
        "conf":   float(probs[pred].item()),
        "probs":  probs.detach().cpu(),   # tensor [2]
        "logits": logits.detach().cpu(),  # tensor [2]
    }

In [6]:
import random
import copy
from cnf_automator import verify_assignment, generate_encoding

r = 7

def add(seq, next_bit):
    first_len = len(seq[0])
    added = False
    for i in range(4):
        if (len(seq[i]) < first_len):
            new_element = (next_bit, )
            new_seq = seq[i] + new_element
            seq[i] = new_seq
            added = True
            break
    if not added:
        new_element = (next_bit, )
        new_seq = seq[0] + new_element
        seq[0] = new_seq
    return seq

def generate_rollout(r):
    seq = [(1, ), (1, ), (1, ), (1, )]
    cnf, varmap, pmap_lags, pool = generate_encoding(r)
    sat, core = verify_assignment(seq, varmap, cnf)

    rev_varmap = {}
    for k, v in varmap.items():
        rev_varmap[v] = k

    model = PolicyOnlyTransformer()
    model.load_state_dict(torch.load(f"model_7_train_test.pt", weights_only=True))
    model.eval()

    rollout_data = []

    for i in range(4 * (r - 1)):
        res = predict_next_bit_from_prefix(model, seq, strategy="sample")
        next_bit = res["pred"]
        prev_seq = copy.deepcopy(seq)
        seq = add(seq, next_bit)
        sec_seq = copy.deepcopy(prev_seq)
        sec_seq = add(sec_seq, 1 - next_bit)
        sat, core = verify_assignment(seq, varmap, cnf)
        sec_sat, sec_core = verify_assignment(sec_seq, varmap, cnf)
        if core and sec_core:
            return rollout_data
        if not core and not sec_core:
            continue
        actual_bit = 1 - next_bit if core else next_bit
        rollout_data.append((prev_seq, actual_bit, 1 - actual_bit))
        if core:
            seq = sec_seq
            continue
    return rollout_data
    

In [7]:
import pickle

def generate_dpo_dataset(r, size, filename):
    file = open(filename, "wb")
    dpo_data = []
    for i in range(size):
        rollout_data = generate_rollout(r)
        dpo_data.extend(rollout_data)
        if i % (size // 10) == 0:
            print(i)
    pickle.dump(dpo_data, file)
    file.close()

In [8]:
generate_dpo_dataset(7, 50, "dpo_test.pkl")

/Users/arnavarora/anaconda3/lib/python3.11/site-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/NestedTensorImpl.cpp:182.)
  output = torch._nested_tensor_from_mask(


0
5
10
15
20
25
30
35
40
45


In [9]:
def collate_pad_dpo(batch):
    """
    batch: list of (x, chosen, rejected)
      x:        [T_i] LongTensor of {0,1}
      chosen:   int in {0,1}
      rejected: int in {0,1}

    returns:
      X:        [B, T] int64 in {0,1,PAD_ID}
      C:        [B]    int64 chosen labels
      R:        [B]    int64 rejected labels
      M:        [B, T] bool   True where PAD
    """
    xs, chosen, rejected = zip(*batch)
    max_len = max(x.size(0) for x in xs)

    padded = []
    masks = []

    for x in xs:
        pad_len = max_len - x.size(0)
        if pad_len > 0:
            x_pad = torch.cat([
                x,
                torch.full((pad_len,), PAD_ID, dtype=torch.long)
            ])
            m_pad = torch.cat([
                torch.zeros(x.size(0), dtype=torch.bool),
                torch.ones(pad_len, dtype=torch.bool)   # True = PAD
            ])
        else:
            x_pad = x
            m_pad = torch.zeros_like(x, dtype=torch.bool)

        padded.append(x_pad)
        masks.append(m_pad)

    X = torch.stack(padded)                       # [B, T]
    M = torch.stack(masks)                        # [B, T] (True = PAD)
    C = torch.tensor(chosen, dtype=torch.long)    # [B]
    R = torch.tensor(rejected, dtype=torch.long)  # [B]

    return X, M, C, R

In [10]:
import torch.nn.functional as F

def dpo_loss_binary(logits, ref_logits, chosen, rejected, beta=0.1):
    """
    logits/ref_logits: [B,2]
    chosen/rejected: [B] token ids in {0,1}
    beta: DPO inverse-temperature (bigger => stronger preference enforcement)
    """
    logp = F.log_softmax(logits, dim=-1)         # [B,2]
    ref_logp = F.log_softmax(ref_logits, dim=-1) # [B,2]

    # log π(chosen|p) - log π(rejected|p)
    pi_gap = logp.gather(1, chosen[:, None]).squeeze(1) - logp.gather(1, rejected[:, None]).squeeze(1)
    ref_gap = ref_logp.gather(1, chosen[:, None]).squeeze(1) - ref_logp.gather(1, rejected[:, None]).squeeze(1)

    # DPO advantage
    delta = pi_gap - ref_gap                     # [B]

    # -log sigmoid(beta * delta)
    return -F.logsigmoid(beta * delta).mean()

In [11]:
import copy
import random
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import DataLoader

def _pref_metrics_from_batch(logits, chosen, rejected):
    """
    logits: [B,2]
    chosen/rejected: [B]
    Returns:
      acc: fraction where logp(chosen) > logp(rejected)
      margin: mean(logp(chosen)-logp(rejected))
    """
    logp = F.log_softmax(logits, dim=-1)
    gap = logp.gather(1, chosen[:, None]).squeeze(1) - logp.gather(1, rejected[:, None]).squeeze(1)
    acc = (gap > 0).float().mean().item()
    margin = gap.mean().item()
    return acc, margin

@torch.no_grad()
def evaluate_dpo(model, dl, device):
    model.eval()
    total_loss = 0.0
    total_acc = 0.0
    total_margin = 0.0
    n_batches = 0

    for x, pad_mask, chosen, rejected in dl:
        x = x.to(device)
        pad_mask = pad_mask.to(device)
        chosen = chosen.to(device)
        rejected = rejected.to(device)

        logits = model(x, pad_mask=pad_mask)  # [B,2]
        # For evaluation, use "implicit" DPO-style preference accuracy/margin.
        acc, margin = _pref_metrics_from_batch(logits, chosen, rejected)

        total_acc += acc
        total_margin += margin
        n_batches += 1

    model.train()
    if n_batches == 0:
        return {"pref_acc": 0.0, "pref_margin": 0.0}
    return {
        "pref_acc": total_acc / n_batches,
        "pref_margin": total_margin / n_batches,
    }

def train_dpo(
    model,
    preference_samples,
    device=None,
    epochs=3,
    batch_size=64,
    lr=3e-4,
    beta=0.1,
    grad_clip=1.0,
    test_frac=0.2,
    seed=0,
):
    """
    Splits preference_samples into train/test, trains with DPO, and evaluates
    preference accuracy & margin on both train/test each epoch.

    Assumes:
      - DPODataset(preference_samples) yields (x, chosen, rejected)
      - collate_pad_dpo returns (X, pad_mask, chosen, rejected)
      - dpo_loss_binary(logits, ref_logits, chosen, rejected, beta) exists
    """
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    rng = random.Random(seed)

    # --- split ---
    idx = list(range(len(preference_samples)))
    rng.shuffle(idx)
    n_test = int(len(idx) * test_frac)
    test_idx = set(idx[:n_test])
    train_samples = [preference_samples[i] for i in idx[n_test:]]
    test_samples  = [preference_samples[i] for i in idx[:n_test]]

    print(f"Split: train={len(train_samples)} test={len(test_samples)} (test_frac={test_frac})")

    # --- models ---
    model = model.to(device)
    model.train()

    # Reference model: frozen copy of the *initial* policy (standard DPO)
    ref_model = copy.deepcopy(model).to(device)
    ref_model.eval()
    for p in ref_model.parameters():
        p.requires_grad_(False)

    # --- loaders ---
    train_ds = DPODataset(train_samples)
    test_ds  = DPODataset(test_samples)

    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                          collate_fn=collate_pad_dpo, drop_last=False)
    # eval loaders: no shuffle
    train_eval_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=False,
                               collate_fn=collate_pad_dpo, drop_last=False)
    test_dl = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                         collate_fn=collate_pad_dpo, drop_last=False)

    opt = AdamW(model.parameters(), lr=lr)

    for epoch in range(1, epochs + 1):
        # --- train epoch ---
        total_loss = 0.0
        n_batches = 0

        for x, pad_mask, chosen, rejected in train_dl:
            x = x.to(device)
            pad_mask = pad_mask.to(device)
            chosen = chosen.to(device)
            rejected = rejected.to(device)

            logits = model(x, pad_mask=pad_mask)  # [B,2]
            with torch.no_grad():
                ref_logits = ref_model(x, pad_mask=pad_mask)  # [B,2]

            loss = dpo_loss_binary(logits, ref_logits, chosen, rejected, beta=beta)

            opt.zero_grad(set_to_none=True)
            loss.backward()
            if grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            opt.step()

            total_loss += float(loss.item())
            n_batches += 1

        avg_loss = total_loss / max(1, n_batches)

        # --- eval each epoch ---
        train_metrics = evaluate_dpo(model, train_eval_dl, device=device)
        test_metrics  = evaluate_dpo(model, test_dl, device=device)

        print(
            f"Epoch {epoch}/{epochs} | "
            f"train_dpo_loss={avg_loss:.4f} | "
            f"train_pref_acc={train_metrics['pref_acc']:.3f} "
            f"train_margin={train_metrics['pref_margin']:.3f} | "
            f"test_pref_acc={test_metrics['pref_acc']:.3f} "
            f"test_margin={test_metrics['pref_margin']:.3f}"
        )

    return model

In [12]:
dpo_filename = "dpo_test.pkl"
with open(dpo_filename, "rb") as file:
    dpo_dataset = pickle.load(file)
model = PolicyOnlyTransformer()
model.load_state_dict(torch.load(f"model_7_train_test.pt", weights_only=True))
model = train_dpo(model=model, preference_samples=dpo_dataset, device="cpu", epochs=10, lr=7e-5, beta=0.4)

Split: train=440 test=110 (test_frac=0.2)
Epoch 1/10 | train_dpo_loss=0.6921 | train_pref_acc=0.513 train_margin=0.103 | test_pref_acc=0.595 test_margin=0.630
Epoch 2/10 | train_dpo_loss=0.6675 | train_pref_acc=0.529 train_margin=0.212 | test_pref_acc=0.615 test_margin=0.641
Epoch 3/10 | train_dpo_loss=0.6539 | train_pref_acc=0.534 train_margin=0.267 | test_pref_acc=0.603 test_margin=0.668
Epoch 4/10 | train_dpo_loss=0.6451 | train_pref_acc=0.531 train_margin=0.350 | test_pref_acc=0.603 test_margin=0.756
Epoch 5/10 | train_dpo_loss=0.6391 | train_pref_acc=0.538 train_margin=0.443 | test_pref_acc=0.600 test_margin=0.808
Epoch 6/10 | train_dpo_loss=0.6267 | train_pref_acc=0.538 train_margin=0.511 | test_pref_acc=0.600 test_margin=0.859
Epoch 7/10 | train_dpo_loss=0.6161 | train_pref_acc=0.551 train_margin=0.561 | test_pref_acc=0.589 test_margin=0.821
Epoch 8/10 | train_dpo_loss=0.6154 | train_pref_acc=0.536 train_margin=0.646 | test_pref_acc=0.581 test_margin=0.953
Epoch 9/10 | train_dpo

In [54]:
torch.save(model.state_dict(), f"model_7_dpo.pt")